# EWS — EDA, Feature Engineering & Model Walkthrough

This notebook walks through the full pipeline:
1. Data overview
2. Behavioral pattern analysis
3. Trajectory feature engineering
4. Model training & evaluation
5. Calibration
6. Operational threshold analysis

**All modeling code lives in . This notebook is for understanding only.**


## 0. Setup

In [ ]:
import sys, json, pickle, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

# Load data
loans    = pd.read_csv('../loans_static.csv')
behavior = pd.read_csv('../behavior_history.csv')
print(f"loans    : {loans.shape}")
print(f"behavior : {behavior.shape}")


## 1. Static Loan Data Overview

In [ ]:
loans.head(3)

In [ ]:
print("=== Basic Stats ===")
print(f"Loans             : {len(loans):,}")
print(f"Date range        : {loans.issue_date.min()} → {loans.issue_date.max()}")
print(f"target_3m default : {loans.target_3m.mean():.2%}  ({loans.target_3m.sum():,} loans)")
print(f"target_6m default : {loans.target_6m.mean():.2%}  ({loans.target_6m.sum():,} loans)")
print()
print(loans[["loan_amt","emi","dti","monthly_income","borrower_age","roi_initial"]].describe().round(2))


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].bar(["Non-default","Default 3m","Default 6m"],
            [loans.target_3m.eq(0).sum(), loans.target_3m.sum(), loans.target_6m.sum()],
            color=["#4CAF50","#FF9800","#F44336"])
axes[0].set_title("Default Counts"); axes[0].set_ylabel("Loans")

loans["loan_amt"].div(1000).hist(bins=40, ax=axes[1], color="#2196F3", edgecolor="none")
axes[1].set_title("Loan Amount (₹000s)"); axes[1].set_xlabel("Amount")

loans["dti"].hist(bins=40, ax=axes[2], color="#9C27B0", edgecolor="none")
axes[2].set_title("Debt-to-Income Ratio")

loans["borrower_age"].hist(bins=30, ax=axes[3], color="#FF5722", edgecolor="none")
axes[3].set_title("Borrower Age")

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

emp_counts = loans["emp_type"].value_counts()
axes[0].bar(emp_counts.index, emp_counts.values, color="#3F51B5")
axes[0].set_title("Employment Type"); axes[0].set_ylabel("Count")

purpose_counts = loans["loan_purpose"].value_counts()
axes[1].barh(purpose_counts.index, purpose_counts.values, color="#009688")
axes[1].set_title("Loan Purpose")

term_counts = loans["term_months"].value_counts().sort_index()
axes[2].bar(term_counts.index.astype(str), term_counts.values, color="#795548")
axes[2].set_title("Loan Term (months)")

plt.tight_layout()
plt.show()


## 2. Behavioral History Analysis

In [ ]:
behavior.head(3)

In [ ]:
print("=== Behavioral Stats ===")
print(f"Total records     : {len(behavior):,}  (15,000 loans × 12 months)")
print(f"Dishonor rate     : {behavior.dishonored.mean():.2%}")
print(f"Mean DPD          : {behavior.days_past_due.mean():.1f} days")
print(f"Max DPD observed  : {behavior.days_past_due.max()} days")
print()
print(behavior[["days_past_due","arrear_balance","payment_amount","effective_roi"]].describe().round(2))


In [ ]:
# DPD trend over 12 months — shows rising risk as we approach observation point
dpd_by_month = behavior.groupby("month_idx")["days_past_due"].mean()
dis_by_month = behavior.groupby("month_idx")["dishonored"].mean() * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(dpd_by_month.index, dpd_by_month.values, marker="o", color="#F44336", linewidth=2)
axes[0].axvline(x=11, color="gray", linestyle="--", alpha=0.5, label="Observation point")
axes[0].set_title("Mean DPD by Month (portfolio-wide)")
axes[0].set_xlabel("Month Index"); axes[0].set_ylabel("Days Past Due")
axes[0].legend(); axes[0].set_xticks(range(12))

axes[1].plot(dis_by_month.index, dis_by_month.values, marker="s", color="#FF9800", linewidth=2)
axes[1].axvline(x=11, color="gray", linestyle="--", alpha=0.5, label="Observation point")
axes[1].set_title("Dishonor Rate by Month (%)")
axes[1].set_xlabel("Month Index"); axes[1].set_ylabel("Dishonor %")
axes[1].legend(); axes[1].set_xticks(range(12))

plt.tight_layout()
plt.show()

print("Key insight: DPD rises from ~9 in month 0 to ~19 in month 11.")
print("This portfolio-wide rise is driven by decliners deteriorating into months 9-11.")


In [ ]:
# DPD distribution — defaulters vs non-defaulters at month 11
beh_m11 = behavior[behavior["month_idx"]==11].merge(loans[["loan_id","target_6m"]], on="loan_id")

fig, ax = plt.subplots(figsize=(12, 4))
ax.hist(beh_m11[beh_m11["target_6m"]==0]["days_past_due"],
        bins=50, alpha=0.6, color="#4CAF50", label="Non-default (6m)", density=True)
ax.hist(beh_m11[beh_m11["target_6m"]==1]["days_past_due"],
        bins=50, alpha=0.6, color="#F44336", label="Default (6m)", density=True)
ax.axvline(x=90, color="black", linestyle="--", linewidth=1.5, label="90 DPD threshold")
ax.set_title("DPD at Month 11 — Defaulters vs Non-Defaulters")
ax.set_xlabel("Days Past Due at Month 11"); ax.set_ylabel("Density")
ax.legend()
plt.tight_layout()
plt.show()

safe = beh_m11[beh_m11["target_6m"]==0]["days_past_due"]
deft = beh_m11[beh_m11["target_6m"]==1]["days_past_due"]
print(f"Non-default median DPD at month 11 : {safe.median():.0f}")
print(f"Default     median DPD at month 11 : {deft.median():.0f}")


In [ ]:
# DPD trajectory: defaulters vs non-defaulters
merged = behavior.merge(loans[["loan_id","target_6m"]], on="loan_id")
dpd_safe = merged[merged["target_6m"]==0].groupby("month_idx")["days_past_due"].mean()
dpd_deft = merged[merged["target_6m"]==1].groupby("month_idx")["days_past_due"].mean()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(dpd_safe.index, dpd_safe.values, color="#4CAF50", linewidth=2.5, marker="o", label="Non-default")
ax.plot(dpd_deft.index, dpd_deft.values, color="#F44336", linewidth=2.5, marker="s", label="Default (6m)")
ax.axvline(x=11, color="gray", linestyle="--", alpha=0.5, label="Observation point")
ax.set_title("Mean DPD Trajectory — Defaulters vs Non-Defaulters")
ax.set_xlabel("Month Index"); ax.set_ylabel("Mean DPD")
ax.set_xticks(range(12)); ax.legend()
plt.tight_layout()
plt.show()

print("Key insight: Defaulters start diverging around month 4-6.")
print("This is the "decliner" regime in action — clean early, deteriorating late.")


## 3. Trajectory Feature Engineering

Six features capture the *shape* of deterioration, not just snapshots.

In [ ]:
from src.features.trajectory import build_trajectory_features

print("Building trajectory features (may take ~10s)...")
traj = build_trajectory_features(behavior)
print(f"Done. Shape: {traj.shape}")
traj.head(3)


In [ ]:
df = loans.merge(traj, on="loan_id")

TRAJ_FEATS = [
    "dpd_slope_3m", "dpd_acceleration", "dishonor_slope_3m",
    "stuck_high_indicator", "arrear_balance_trend", "payment_gap_variance"
]

# Mean by target_6m
print("Feature means — Non-default vs Default (6m horizon):")
print("-" * 65)
print(f"{'Feature':<28} {'Non-default':>14} {'Default':>12}  {'Ratio':>6}")
print("-" * 65)
for f in TRAJ_FEATS:
    m0 = df.loc[df["target_6m"]==0, f].mean()
    m1 = df.loc[df["target_6m"]==1, f].mean()
    ratio = m1/m0 if m0 != 0 else float("inf")
    print(f"{f:<28} {m0:>14.3f} {m1:>12.3f}  {ratio:>6.1f}x")


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, feat in enumerate(TRAJ_FEATS):
    safe_vals = df.loc[df["target_6m"]==0, feat].clip(
        df[feat].quantile(0.01), df[feat].quantile(0.99))
    deft_vals = df.loc[df["target_6m"]==1, feat].clip(
        df[feat].quantile(0.01), df[feat].quantile(0.99))

    axes[i].hist(safe_vals, bins=40, alpha=0.6, color="#4CAF50",
                 label="Non-default", density=True)
    axes[i].hist(deft_vals, bins=40, alpha=0.6, color="#F44336",
                 label="Default", density=True)
    axes[i].set_title(feat, fontsize=10)
    axes[i].legend(fontsize=8)

plt.suptitle("Trajectory Feature Distributions — Default vs Non-Default", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# Correlation of trajectory features with target
corr = df[TRAJ_FEATS + ["target_3m","target_6m"]].corr()
traj_corr = corr[["target_3m","target_6m"]].drop(["target_3m","target_6m"])

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(TRAJ_FEATS))
width = 0.35
ax.bar(x - width/2, traj_corr["target_3m"], width, label="target_3m", color="#FF9800", alpha=0.8)
ax.bar(x + width/2, traj_corr["target_6m"], width, label="target_6m", color="#F44336", alpha=0.8)
ax.set_xticks(x); ax.set_xticklabels(TRAJ_FEATS, rotation=30, ha="right")
ax.set_title("Pearson Correlation of Trajectory Features with Target")
ax.axhline(0, color="black", linewidth=0.8); ax.legend()
plt.tight_layout()
plt.show()


## 4. Model Training

**Two separate LightGBM classifiers** — one per horizon.

- Split: sorted by , first 70% → train, last 30% → test (no random split)
- Within train: last 20% held out for isotonic calibration
-  handles the ~11%/15% imbalance
- No resampling needed per spec

In [ ]:
# Visualise the time-based split
loans_sorted = loans.sort_values("issue_date")
split_idx = int(len(loans_sorted) * 0.70)
cal_idx   = int(split_idx * 0.80)

train_dates = pd.to_datetime(loans_sorted.iloc[:cal_idx]["issue_date"])
cal_dates   = pd.to_datetime(loans_sorted.iloc[cal_idx:split_idx]["issue_date"])
test_dates  = pd.to_datetime(loans_sorted.iloc[split_idx:]["issue_date"])

fig, ax = plt.subplots(figsize=(14, 3))
ax.barh(0, len(train_dates), left=0,       height=0.5, color="#2196F3", label=f"Train ({len(train_dates):,})")
ax.barh(0, len(cal_dates),   left=len(train_dates), height=0.5, color="#FF9800", label=f"Calibration ({len(cal_dates):,})")
ax.barh(0, len(test_dates),  left=split_idx, height=0.5, color="#4CAF50", label=f"Test ({len(test_dates):,})")
ax.set_xlim(0, len(loans))
ax.set_yticks([]); ax.set_xlabel("Loan index (sorted by issue_date)")
ax.set_title("Train / Calibration / Test Split (time-ordered)")
ax.legend(loc="upper right")

# Annotate date boundaries
ax.text(cal_idx//2, 0.32, f"{train_dates.min().date()}
→ {train_dates.max().date()}",
        ha="center", fontsize=8)
ax.text(cal_idx + len(cal_dates)//2, 0.32, f"Cal", ha="center", fontsize=8)
ax.text(split_idx + len(test_dates)//2, 0.32, f"{test_dates.min().date()}
→ {test_dates.max().date()}",
        ha="center", fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
# Load trained models and metadata
model3  = pickle.load(open("../models/lgbm_3m.pkl", "rb"))
model6  = pickle.load(open("../models/lgbm_6m.pkl", "rb"))
cal3    = pickle.load(open("../models/calibrator_3m.pkl", "rb"))
cal6    = pickle.load(open("../models/calibrator_6m.pkl", "rb"))
meta    = json.load(open("../models/metadata.json"))

print("Model: LGBMClassifier")
print("  n_estimators   :", model3.get_params()["n_estimators"])
print("  learning_rate  :", model3.get_params()["learning_rate"])
print("  num_leaves     :", model3.get_params()["num_leaves"])
print("  class_weight   :", model3.get_params()["class_weight"])
print()
print("Two separate models: model_3m predicts target_3m, model_6m predicts target_6m.")
print("Rationale: each horizon has different signal strengths — trajectory features")
print("are more predictive at 6m (more time for patterns to emerge) vs 3m.")


## 5. Feature Importance

In [ ]:
fn = meta["feature_names"]

imp3 = sorted(zip(fn, model3.feature_importances_), key=lambda x: -x[1])[:12]
imp6 = sorted(zip(fn, model6.feature_importances_), key=lambda x: -x[1])[:12]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, imp, title, color in [
    (axes[0], imp3, "model_3m — Top 12 Features", "#FF9800"),
    (axes[1], imp6, "model_6m — Top 12 Features", "#F44336"),
]:
    names = [x[0] for x in imp][::-1]
    vals  = [x[1] for x in imp][::-1]
    bars = ax.barh(names, vals, color=color, alpha=0.8)
    ax.set_title(title)
    ax.set_xlabel("Importance (split count)")
    # Highlight trajectory features
    for bar, name in zip(bars, names):
        if name in ["dpd_slope_3m","dpd_acceleration","dishonor_slope_3m",
                    "stuck_high_indicator","arrear_balance_trend","payment_gap_variance"]:
            bar.set_edgecolor("#1A237E"); bar.set_linewidth(2)

plt.suptitle("Feature Importance (blue border = trajectory feature)", fontsize=12)
plt.tight_layout()
plt.show()


## 6. Model Evaluation

In [ ]:
from sklearn.metrics import roc_curve, auc as sk_auc, brier_score_loss

EMP_TYPE_MAP     = {"E":0,"S":1,"M":2,"R":3}
LOAN_PURPOSE_MAP = {"FLAT_PURCH":0,"CONSTRUCTION":1,"PLOT_PURCH":2,"RENOVATION":3,"TAKEOVER":4}
PROPERTY_TYPE_MAP= {"FLAT":0,"BUNGALOW":1,"PLOT":2,"COMMERCIAL":3}
RATE_TYPE_MAP    = {"A":0,"F":1}
LOAN_SCHEME_MAP  = {"1":0,"2":1}

df_enc = df.copy()
df_enc["emp_type_enc"]      = df_enc["emp_type"].map(EMP_TYPE_MAP)
df_enc["loan_purpose_enc"]  = df_enc["loan_purpose"].map(LOAN_PURPOSE_MAP)
df_enc["property_type_enc"] = df_enc["property_type"].map(PROPERTY_TYPE_MAP)
df_enc["rate_type_enc"]     = df_enc["rate_type"].map(RATE_TYPE_MAP)
df_enc["loan_scheme_enc"]   = df_enc["loan_scheme"].map(LOAN_SCHEME_MAP)

df_sorted = df_enc.sort_values("issue_date").reset_index(drop=True)
split_idx = int(len(df_sorted)*0.70)
test_df   = df_sorted.iloc[split_idx:]
X_test    = test_df[fn].values
y3 = test_df["target_3m"].values
y6 = test_df["target_6m"].values

raw3  = model3.predict_proba(X_test)[:,1]
raw6  = model6.predict_proba(X_test)[:,1]
cal3p = cal3.transform(raw3)
cal6p = cal6.transform(raw6)

print("Test-set metrics (calibrated probabilities):")
print("-"*52)
print(f"{'Metric':<26} {'model_3m':>12} {'model_6m':>12}")
print("-"*52)

m3 = meta["metrics"]["3m"]
m6 = meta["metrics"]["6m"]
rows = [
    ("AUC-ROC",         f"{m3['auc']:.4f}",     f"{m6['auc']:.4f}"),
    ("Brier (raw)",     f"{m3['brier_raw']:.4f}", f"{m6['brier_raw']:.4f}"),
    ("Brier (calib.)",  f"{m3['brier']:.4f}",    f"{m6['brier']:.4f}"),
    ("Precision@Top10%",f"{m3['prec10']:.4f}",   f"{m6['prec10']:.4f}"),
    ("Recall@Top10%",   f"{m3['rec10']:.4f}",    f"{m6['rec10']:.4f}"),
]
for label, v3, v6 in rows:
    print(f"{label:<26} {v3:>12} {v6:>12}")


In [ ]:
# ROC curves
fpr3, tpr3, _ = roc_curve(y3, cal3p)
fpr6, tpr6, _ = roc_curve(y6, cal6p)
auc3 = sk_auc(fpr3, tpr3)
auc6 = sk_auc(fpr6, tpr6)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, fpr, tpr, auc_val, title, color in [
    (axes[0], fpr3, tpr3, auc3, "model_3m ROC Curve", "#FF9800"),
    (axes[1], fpr6, tpr6, auc6, "model_6m ROC Curve", "#F44336"),
]:
    ax.plot(fpr, tpr, color=color, linewidth=2.5, label=f"AUC = {auc_val:.4f}")
    ax.plot([0,1],[0,1], linestyle="--", color="gray", linewidth=1)
    ax.fill_between(fpr, tpr, alpha=0.1, color=color)
    ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
    ax.set_title(title); ax.legend(loc="lower right")

plt.suptitle("ROC Curves — Test Set (calibrated)", fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# Precision-Recall at different top-K thresholds
def pr_at_topk(y_true, y_prob, k_pct):
    n = len(y_true)
    k = max(1, int(n * k_pct))
    top_idx = np.argsort(y_prob)[-k:]
    tp = y_true[top_idx].sum()
    return tp/k, tp/max(1, y_true.sum())

k_pcts = np.arange(0.01, 0.31, 0.01)
prec3 = [pr_at_topk(y3, cal3p, k)[0] for k in k_pcts]
rec3  = [pr_at_topk(y3, cal3p, k)[1] for k in k_pcts]
prec6 = [pr_at_topk(y6, cal6p, k)[0] for k in k_pcts]
rec6  = [pr_at_topk(y6, cal6p, k)[1] for k in k_pcts]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
k_labels = [f"{int(k*100)}%" for k in k_pcts]

ax = axes[0]
ax.plot(k_pcts*100, prec3, color="#FF9800", linewidth=2, label="Precision 3m")
ax.plot(k_pcts*100, rec3,  color="#FF9800", linewidth=2, linestyle="--", label="Recall 3m")
ax.plot(k_pcts*100, prec6, color="#F44336", linewidth=2, label="Precision 6m")
ax.plot(k_pcts*100, rec6,  color="#F44336", linewidth=2, linestyle="--", label="Recall 6m")
ax.axvline(x=200/150, color="black", linestyle=":", label="200 calls/week budget (1.33%)")
ax.set_xlabel("Top K% flagged"); ax.set_ylabel("Score")
ax.set_title("Precision & Recall at Top-K%"); ax.legend(fontsize=8)

ax = axes[1]
ax.plot(rec3, prec3, color="#FF9800", linewidth=2.5, label="model_3m")
ax.plot(rec6, prec6, color="#F44336", linewidth=2.5, label="model_6m")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curve"); ax.legend()

plt.suptitle("Operational Performance", fontsize=13)
plt.tight_layout()
plt.show()


## 7. Calibration

Isotonic regression is applied to map raw LightGBM scores to proper probabilities.

In [ ]:
# Score distribution — raw vs calibrated
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for row, (raw, calp, y, label) in enumerate([
    (raw3, cal3p, y3, "3m"),
    (raw6, cal6p, y6, "6m"),
]):
    # Distribution
    axes[row][0].hist(raw[y==0],  bins=40, alpha=0.6, color="#4CAF50", label="Non-default", density=True)
    axes[row][0].hist(raw[y==1],  bins=40, alpha=0.6, color="#F44336", label="Default",     density=True)
    axes[row][0].set_title(f"Raw Scores — model_{label}")
    axes[row][0].set_xlabel("Score"); axes[row][0].legend()

    axes[row][1].hist(calp[y==0], bins=40, alpha=0.6, color="#4CAF50", label="Non-default", density=True)
    axes[row][1].hist(calp[y==1], bins=40, alpha=0.6, color="#F44336", label="Default",     density=True)
    axes[row][1].set_title(f"Calibrated Probabilities — model_{label}")
    axes[row][1].set_xlabel("Probability"); axes[row][1].legend()

plt.suptitle("Score Distributions Before/After Isotonic Calibration", fontsize=13)
plt.tight_layout()
plt.show()

print("Brier improvement from calibration:")
for label, raw, calp, y in [("3m",raw3,cal3p,y3),("6m",raw6,cal6p,y6)]:
    b_raw = brier_score_loss(y, raw)
    b_cal = brier_score_loss(y, calp)
    print(f"  model_{label}: {b_raw:.4f} → {b_cal:.4f}  ({(b_raw-b_cal)/b_raw:.1%} improvement)")


## 8. Operational Threshold

Constraint: 200 calls/week from 15,000-loan portfolio = **1.33% flag rate**.

In [ ]:
n_test = len(y3)
budget = int(round(200 * n_test / 15000))
print(f"Test set size   : {n_test:,}")
print(f"Scaled budget   : {budget} loans to flag")
print()

for label, calp, y in [("3m", cal3p, y3), ("6m", cal6p, y6)]:
    thresh = float(np.sort(calp)[-budget] - 1e-9)
    flagged = (calp >= thresh).sum()
    tp = ((calp >= thresh) & (y == 1)).sum()
    prec = tp / max(1, flagged)
    rec  = tp / max(1, y.sum())
    print(f"model_{label}:")
    print(f"  Threshold      : {thresh:.4f}")
    print(f"  Flagged        : {flagged} ({flagged/n_test:.2%} of test)")
    print(f"  True Positives : {tp}")
    print(f"  Precision      : {prec:.4f}")
    print(f"  Recall         : {rec:.4f}")
    print()


In [ ]:
# Score histogram with operational threshold marked
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, calp, y, label, color in [
    (axes[0], cal3p, y3, "model_3m", "#FF9800"),
    (axes[1], cal6p, y6, "model_6m", "#F44336"),
]:
    n_test = len(y)
    budget = int(round(200 * n_test / 15000))
    thresh = float(np.sort(calp)[-budget] - 1e-9)

    ax.hist(calp[y==0], bins=40, alpha=0.6, color="#4CAF50", label="Non-default", density=True)
    ax.hist(calp[y==1], bins=40, alpha=0.6, color=color,    label="Default",     density=True)
    ax.axvline(x=thresh, color="black", linewidth=2, linestyle="--",
               label=f"Op. threshold={thresh:.3f}")
    ax.set_title(f"{label} — Calibrated Score Distribution")
    ax.set_xlabel("P(default)"); ax.legend(fontsize=8)

plt.suptitle("Operational Threshold (200 calls/week budget)", fontsize=13)
plt.tight_layout()
plt.show()


## 9. Summary

| Component | Choice | Reason |
|---|---|---|
| Model | LightGBM (2 separate) | Fast, handles mixed types, interpretable importances |
| Split | Time-ordered 70/30 | Prevents data leakage across time |
| Calibration | Isotonic regression | Tree scores are bimodal, not sigmoid |
| Imbalance |  | No resampling — per spec |
| Horizon | Two heads | Signal strengths differ at 3m vs 6m |

**Key trajectory features** (by predictive power):
1.  — overall DPD level; highest importance in both models
2.  — rate-of-change of deterioration
3.  — snapshot at observation point; DPD already crossing 90 is a near-certain signal
4.  — trend in final 3 months
5.  — erratic payment behaviour

**Why 6m AUC > 3m AUC on this data**: Decliners deteriorate steeply enough that by month 11 their trajectory features are already extreme — making 6m targets easier to predict than 3m targets, which require the loan to cross 90 DPD within just 3 months of observation.
